# 02 · Batch inference

Runs `run_inference_batch` over the prepared Delta table, writes
`inference_results`, logs metrics to MLflow, and visualizes the output.

In [ ]:
# ---- CONFIG (the only cell you must edit) --------------------------------
dbutils.widgets.text("input_delta_path", "drone.tracks", "Input Delta table/path")
dbutils.widgets.text("output_delta_path", "drone.inference_results", "Output Delta table/path")
dbutils.widgets.text("model_name", "visdrone-yolov8x", "DVSA model name")
dbutils.widgets.text("batch_size", "64", "Batch size")
dbutils.widgets.text("checkpoint_location", "/Volumes/drone/_chk", "Checkpoint (streaming)")
dbutils.widgets.text("secret_scope", "dvsa", "Databricks Secret scope")

INPUT_DELTA = dbutils.widgets.get("input_delta_path")
OUTPUT_DELTA = dbutils.widgets.get("output_delta_path")
MODEL_NAME = dbutils.widgets.get("model_name")
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
CHECKPOINT = dbutils.widgets.get("checkpoint_location")
SECRET_SCOPE = dbutils.widgets.get("secret_scope")

In [ ]:
from dvsa_databricks_connector import config_from_secrets, prepare_context_from_delta, run_inference_batch

cfg = config_from_secrets(dbutils, scope=SECRET_SCOPE)
df = prepare_context_from_delta(spark, INPUT_DELTA)

results = run_inference_batch(
    df,
    model_name=MODEL_NAME,
    config=cfg,
    batch_size=BATCH_SIZE,
    output_table=OUTPUT_DELTA,
    mlflow_run=True,
)
display(results)

In [ ]:
# Quick roll-up of detections per model.
spark.read.table(OUTPUT_DELTA).groupBy("model_name").sum("num_detections").show()